# Placement Readiness Intelligence System (PRIS)

Complete pipeline: synthetic dataset -> trained ML model -> Groq-powered resume/JD analysis -> feature builder -> readiness prediction -> LLM feedback.

Run the cells **in order, top to bottom**.

## Step 1 — Load the Synthetic Readiness Dataset

The dataset (11 features, generated with per-class overlapping ranges + noise + label noise) is already generated and committed to GitHub, so it's loaded directly here rather than regenerated each run.

If you ever need to regenerate it from scratch (e.g. to change class ranges or feature definitions), use the separate `step1_dataset_generator.py` script, then re-upload the resulting CSV to GitHub.

In [ ]:
import pandas as pd

dataset_url = "https://raw.githubusercontent.com/sravanioffice1997-arch/Placement-Readiness-Intelligence-System/main/Source/placement_readiness_dataset.csv"
df = pd.read_csv(dataset_url)

print("Shape:", df.shape)
print("\nClass distribution:")
print(df["readiness_label"].value_counts())
df.head()


## Step 2 — Train & Compare ML Models

Trains Logistic Regression, Decision Tree, Random Forest, and Gradient Boosting; picks the best by test accuracy; saves the model + label encoder + feature column order to `.pkl` files.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

# ---- df is already loaded from GitHub in Step 1 above ----

feature_cols = [c for c in df.columns if c != "readiness_label"]
X = df[feature_cols]
y = df["readiness_label"]

# ---- Encode labels (models need numbers, not strings) ----
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
print("Label mapping:", dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

# ---- Train/test split ----
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# ---- Define the 4 candidate models ----
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=8, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=150, max_depth=4, random_state=42),
}

# ---- Train each, evaluate on test set ----
results = {}
trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    results[name] = acc
    trained_models[name] = model
    print(f"\n{'='*50}")
    print(f"{name} — Accuracy: {acc:.4f}")
    print(classification_report(y_test, preds, target_names=label_encoder.classes_))

# ---- Pick the best model ----
best_name = max(results, key=results.get)
best_model = trained_models[best_name]
print(f"\n{'='*50}")
print(f"BEST MODEL: {best_name} (accuracy = {results[best_name]:.4f})")
print(f"{'='*50}")

# ---- Confusion matrix for the winner ----
best_preds = best_model.predict(X_test)
cm = confusion_matrix(y_test, best_preds)
print("\nConfusion Matrix ({}):".format(best_name))
print(pd.DataFrame(cm, index=label_encoder.classes_, columns=label_encoder.classes_))

# ---- Feature importance (if the winner supports it) ----
if hasattr(best_model, "feature_importances_"):
    importance_df = pd.DataFrame({
        "feature": feature_cols,
        "importance": best_model.feature_importances_
    }).sort_values("importance", ascending=False)
    print("\nFeature Importances:")
    print(importance_df)

# ---- Save the winning model + label encoder + feature column order ----
joblib.dump(best_model, "readiness_model.pkl")
joblib.dump(label_encoder, "label_encoder.pkl")
joblib.dump(feature_cols, "feature_columns.pkl")

print("\nSaved: readiness_model.pkl, label_encoder.pkl, feature_columns.pkl")


## Step 3 — Resume + JD Analyzer using Groq API

Contextual extraction — no fixed skill list. Splits JD skills into required vs optional based on the JD's own structure, and matches semantically (including evidence found in project descriptions).

In [ ]:
!pip install groq PyPDF2 -q

from groq import Groq
import PyPDF2
import json
import re

# ---- Paste your Groq API key here each session. Do NOT save the notebook ----
# ---- with the real key still in this cell -- clear it before saving.     ----
GROQ_API_KEY = "PASTE_YOUR_GROQ_API_KEY_HERE"
client = Groq(api_key=GROQ_API_KEY)

def extract_resume_text(pdf_path):
    text = ""
    with open(pdf_path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            text += page.extract_text() + "\n"
    return text.strip()

def call_groq(prompt, temperature=0.2):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
    )
    raw = response.choices[0].message.content.strip()
    raw = re.sub(r"^```json|```$", "", raw, flags=re.MULTILINE).strip()
    return json.loads(raw)

def analyze_resume(resume_text):
    prompt = f'''
You are an expert technical recruiter. Read the resume text below and extract structured information.
Do NOT rely on any fixed skill list — infer skills contextually from projects, experience, and tools mentioned.

Return ONLY valid JSON, no markdown, no extra text, in this exact structure:
{{
  "skills": ["list", "of", "all", "technical", "and", "soft", "skills", "found"],
  "projects": ["short description of each project"],
  "certifications": ["list of certifications, empty list if none"],
  "internships": ["list of internships/work experience, empty list if none"],
  "education": "highest degree and field",
  "tools_and_technologies": ["list of tools/frameworks/languages"],
  "resume_completeness_score": <integer 0-100, based on how complete/detailed the resume is>
}}

Resume text:
"""{resume_text}"""
'''
    return call_groq(prompt, temperature=0)

def analyze_jd(jd_text):
    prompt = f'''
You are an expert technical recruiter. Read the job description below and extract structured requirements.
Do NOT rely on any fixed skill list — infer required skills contextually.

IMPORTANT — how to classify skills:
- If the JD has explicit sections like "Required Skills" / "Must Have" / "Essential" —
  put those skills in "required_skills".
- If the JD has explicit sections like "Good to Have" / "Nice to Have" / "Preferred" / "Bonus" —
  put those skills in "optional_skills", NOT in "required_skills".
- Within "required_skills", mark a skill as "critical" only if the JD phrasing signals it's
  non-negotiable (e.g. "must have", "strong experience in", "proficiency in", listed first,
  core to the role). Mark supporting/secondary required skills as required but not critical.
- If the JD has no explicit Required/Good-to-Have structure, use your judgment based on
  emphasis and phrasing.

Return ONLY valid JSON, no markdown, no extra text, in this exact structure:
{{
  "job_title": "string",
  "role_category": "string (e.g. Data Science, Web Development, DevOps)",
  "required_skills": ["all required/must-have skills, from explicit Required section if present"],
  "critical_skills": ["subset of required_skills that are absolutely essential/non-negotiable"],
  "optional_skills": ["all Good-to-Have / Preferred / Bonus skills, kept separate from required_skills"],
  "tools_and_frameworks": ["list"],
  "soft_skills": ["list"],
  "responsibilities": ["list of key responsibilities"]
}}

Job description text:
"""{jd_text}"""
'''
    return call_groq(prompt, temperature=0)

def analyze_skill_gap(resume_data, jd_data):
    prompt = f'''
Compare the candidate's resume profile with the job requirements below.
Match skills semantically and generously, for BOTH required and optional skills:
- "Statistics" satisfies "statistical modeling" or "statistics and probability"
- Training/evaluating ML models (e.g. "trained XGBoost and Random Forest models") satisfies "model evaluation" even if not explicitly named as a skill
- "React.js" matches "React", "ML" matches "Machine Learning", etc.
- If a skill is demonstrated through a PROJECT even if not listed as a standalone skill, count it as present — this applies equally to optional skills. For example, if a project mentions "Deep Learning & NLP" or "sentiment analysis model", that counts as evidence for deep learning frameworks and/or NLP even if not listed under resume skills.
- Read project descriptions carefully for technology and domain mentions, not just the skills list.

Resume skills: {resume_data['skills']}
Resume tools: {resume_data['tools_and_technologies']}
Resume projects: {resume_data['projects']}
JD required skills: {jd_data['required_skills']}
JD critical skills: {jd_data['critical_skills']}
JD optional/good-to-have skills: {jd_data.get('optional_skills', [])}

Return ONLY valid JSON, no markdown, in this exact structure:
{{
  "matched_skills": ["required skills present, including those demonstrated via projects"],
  "missing_skills": ["required skills genuinely not found anywhere in resume/skills/projects"],
  "critical_missing_skills": ["critical skills not found"],
  "matched_optional_skills": ["optional/good-to-have skills the candidate does have"],
  "missing_optional_skills": ["optional/good-to-have skills the candidate does NOT have"],
  "skills_to_learn_first": ["top 3-5 priority skills to learn, required skills first"]
}}
'''
    return call_groq(prompt, temperature=0)

print("Step 3 functions loaded: extract_resume_text, analyze_resume, analyze_jd, analyze_skill_gap")


### Step 3 Test Run

Downloads the resume directly from GitHub (no manual upload) and runs it against a sample JD.

In [ ]:
import urllib.request

resume_url = "https://raw.githubusercontent.com/sravanioffice1997-arch/Placement-Readiness-Intelligence-System/main/Source/pathipakasravani_pdf.pdf"
resume_path = "resume.pdf"
urllib.request.urlretrieve(resume_url, resume_path)

resume_text = extract_resume_text(resume_path)

sample_jd = """
We are hiring a Data Analyst with experience in Python, SQL, Power BI, and statistics.
Familiarity with machine learning basics is a plus. Must have strong communication skills.
"""

resume_data = analyze_resume(resume_text)
jd_data = analyze_jd(sample_jd)
gap_data = analyze_skill_gap(resume_data, jd_data)

print("RESUME DATA:", json.dumps(resume_data, indent=2))
print("\nJD DATA:", json.dumps(jd_data, indent=2))
print("\nSKILL GAP:", json.dumps(gap_data, indent=2))


## Step 4 — Feature Builder + ML Prediction

Converts the Step 3 outputs into the 11 numeric features (JD-anchored, with `optional_skill_match_percentage` as its own independent feature) and predicts readiness using the model trained in Step 2.

In [ ]:
import joblib
import numpy as np

# ---- Load the trained model artifacts from Step 2 ----
model = joblib.load("readiness_model.pkl")
label_encoder = joblib.load("label_encoder.pkl")
feature_cols = joblib.load("feature_columns.pkl")

def build_features(resume_data, jd_data, gap_data):
    """
    Converts LLM outputs into the 11 numeric features.
    RULE: all percentage/match scores are computed relative to the JD's
    requirements (denominator = JD), never relative to the resume's total
    skill count. Extra resume skills not asked for by the JD do NOT inflate
    the score. Trusts gap_data's semantic matching (from the LLM comparison)
    instead of doing brittle literal string matching.
    """
    required = jd_data.get("required_skills", []) or []
    critical = jd_data.get("critical_skills", []) or []
    optional = jd_data.get("optional_skills", []) or []
    matched = gap_data.get("matched_skills", []) or []
    missing = gap_data.get("missing_skills", []) or []
    critical_missing = gap_data.get("critical_missing_skills", []) or []
    matched_optional = gap_data.get("matched_optional_skills", []) or []

    # skill_match_percentage: matched / JD-required (JD-anchored)
    skill_match_percentage = round((len(matched) / len(required)) * 100) if required else 0

    # critical_skill_match_percentage
    matched_lower = [m.lower() for m in matched]
    critical_matched = [s for s in critical if s.lower() in matched_lower]
    critical_skill_match_percentage = round((len(critical_matched) / len(critical)) * 100) if critical else 100

    missing_skills_count = len(missing)
    critical_missing_skills_count = len(critical_missing)

    # optional_skill_match_percentage: its OWN feature (11th feature), so the
    # trained model learns the right weight for it instead of us forcing it
    # into another feature. Missing optional skills should NOT gate a
    # candidate out of "Highly Ready" if all required skills are met.
    optional_skill_match_percentage = round((len(matched_optional) / len(optional)) * 100) if optional else 100

    # keyword_match_score: represents required-skill match strength
    keyword_match_score = skill_match_percentage

    # project_relevance_score: fraction of MATCHED skills backed by evidence
    # in projects/tools, not literal phrase-matching
    project_text = " ".join(resume_data.get("projects", [])).lower()
    tools_text = " ".join(resume_data.get("tools_and_technologies", [])).lower()
    evidence_text = project_text + " " + tools_text
    if matched:
        supported = sum(1 for s in matched if any(word in evidence_text for word in s.lower().split()))
        project_relevance_score = round((supported / len(matched)) * 100)
    else:
        project_relevance_score = 40
    num_projects = len(resume_data.get("projects", []))
    project_relevance_score = min(100, project_relevance_score + min(num_projects * 3, 15))

    certs = resume_data.get("certifications", []) or []
    certification_relevance_score = 70 if certs else 20

    internships = resume_data.get("internships", []) or []
    internship_relevance_score = 70 if internships else 20

    resume_completeness_score = resume_data.get("resume_completeness_score", 50)

    # role_category_match_score: based on overall skill match strength
    if skill_match_percentage >= 80:
        role_category_match_score = 90
    elif skill_match_percentage >= 60:
        role_category_match_score = 70
    elif skill_match_percentage >= 40:
        role_category_match_score = 50
    else:
        role_category_match_score = 25

    return {
        "skill_match_percentage": skill_match_percentage,
        "critical_skill_match_percentage": critical_skill_match_percentage,
        "missing_skills_count": missing_skills_count,
        "critical_missing_skills_count": critical_missing_skills_count,
        "optional_skill_match_percentage": optional_skill_match_percentage,
        "project_relevance_score": project_relevance_score,
        "certification_relevance_score": certification_relevance_score,
        "internship_relevance_score": internship_relevance_score,
        "resume_completeness_score": resume_completeness_score,
        "keyword_match_score": keyword_match_score,
        "role_category_match_score": role_category_match_score,
    }


def predict_readiness(resume_data, jd_data, gap_data):
    features = build_features(resume_data, jd_data, gap_data)

    X = np.array([[features[col] for col in feature_cols]])

    pred_encoded = model.predict(X)[0]
    pred_label = label_encoder.inverse_transform([pred_encoded])[0]
    pred_proba = model.predict_proba(X)[0]

    class_order = ["Not Ready Yet", "Needs Improvement", "Moderately Ready", "Highly Ready"]
    label_to_rank = {l: i for i, l in enumerate(class_order)}
    proba_dict = dict(zip(label_encoder.inverse_transform(range(len(pred_proba))), pred_proba))

    weighted_rank = sum(label_to_rank[label] * prob for label, prob in proba_dict.items())
    readiness_score = round((weighted_rank / (len(class_order) - 1)) * 100)
    # Cap at 90: no resume is truly "perfect" against a JD, so we reserve
    # 91-100 to avoid implying flawlessness even at maximum model confidence.
    readiness_score = min(readiness_score, 90)

    return {
        "features": features,
        "readiness_label": pred_label,
        "readiness_score": readiness_score,
        "class_probabilities": proba_dict,
    }


# ---- Run it on the resume_data / jd_data / gap_data from the Step 3 test run ----
result = predict_readiness(resume_data, jd_data, gap_data)

print("FEATURES BUILT:")
for k, v in result["features"].items():
    print(f"  {k}: {v}")

print(f"\nPlacement Readiness Score: {result['readiness_score']} / 100")
print(f"Readiness Level: {result['readiness_label']}")
print(f"\nClass Probabilities:")
for label, prob in result["class_probabilities"].items():
    print(f"  {label}: {prob*100:.1f}%")


## Step 5 — LLM-Powered Feedback & Improvement Plan Generator

Sends the resume, JD, skill gap, and ML prediction to Groq and asks for a structured coaching report.

In [ ]:
def generate_feedback(resume_data, jd_data, gap_data, prediction_result):
    prompt = f'''
You are a career coach helping a student prepare for a job application.

Job Title: {jd_data.get('job_title')}
Role Category: {jd_data.get('role_category')}

Placement Readiness Score: {prediction_result['readiness_score']}/100
Readiness Level: {prediction_result['readiness_label']}

Matched Skills: {gap_data.get('matched_skills')}
Missing Skills: {gap_data.get('missing_skills')}
Critical Missing Skills: {gap_data.get('critical_missing_skills')}

Candidate's Projects: {resume_data.get('projects')}
Candidate's Certifications: {resume_data.get('certifications')}
Candidate's Education: {resume_data.get('education')}

Based on this, generate a coaching report. Return ONLY valid JSON, no markdown, in this exact structure:
{{
  "summary": "10-20 line easy-to-read summary explaining whether this student is suitable for this job role, written directly to the student in second person",
  "strengths": ["list of 3-5 specific strengths based on their actual resume"],
  "gaps": ["list of specific gaps, referencing missing_skills if any, otherwise note minor improvement areas"],
  "seven_day_plan": ["list of 4-6 concrete, specific daily/short-term action items"],
  "thirty_day_plan": ["list of 4-6 concrete, longer-term action items"],
  "resume_improvement_suggestions": ["list of 3-5 specific suggestions to improve the resume itself"],
  "interview_tips": ["list of 3-5 tips specific to this role and this candidate's background"]
}}
'''
    return call_groq(prompt, temperature=0.4)


# ---- Run it ----
feedback = generate_feedback(resume_data, jd_data, gap_data, result)

print("SUMMARY:\n", feedback["summary"])
print("\nSTRENGTHS:")
for s in feedback["strengths"]: print(" -", s)
print("\nGAPS:")
for g in feedback["gaps"]: print(" -", g)
print("\n7-DAY PLAN:")
for d in feedback["seven_day_plan"]: print(" -", d)
print("\n30-DAY PLAN:")
for d in feedback["thirty_day_plan"]: print(" -", d)
print("\nRESUME IMPROVEMENT SUGGESTIONS:")
for r in feedback["resume_improvement_suggestions"]: print(" -", r)
print("\nINTERVIEW TIPS:")
for t in feedback["interview_tips"]: print(" -", t)
